# 02 — Network Construction

## 1. Objective & Inputs
Xây dựng và kiểm tra mạng đồng mua vô hướng, có trọng số từ hai file Parquet của Phase 1. Chọn sản phẩm theo số hóa đơn chứa sản phẩm; lưu toàn bộ pair counts trước khi lọc cạnh. Các tham số trong `config.py` là cấu hình ban đầu, chưa được tối ưu.

Không cleaning lại, không loại hóa đơn lớn khỏi baseline, không thực hiện centrality/community detection hoặc tính chỉ số bán chéo. Chạy từ project root hoặc `notebooks/` với kernel của `.venv`.

In [13]:
from pathlib import Path
from collections import Counter
from itertools import combinations
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import networkx as nx
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
assert (project_root / "config.py").is_file(), "Run from project root or notebooks/."
sys.path.insert(0, str(project_root))
from config import (
    PROJECT_ROOT, PROCESSED_DATA_DIR, TABLES_DIR, NETWORKS_DIR,
    TOP_N_PRODUCTS, MIN_EDGE_WEIGHT, RANDOM_STATE,
)
assert type(TOP_N_PRODUCTS) is int and 0 < TOP_N_PRODUCTS <= 100, "Phase 2 supports TOP_N_PRODUCTS from 1 to 100."
assert type(MIN_EDGE_WEIGHT) is int and MIN_EDGE_WEIGHT > 0, "MIN_EDGE_WEIGHT must be a positive integer."
np.random.seed(RANDOM_STATE)
input_paths = {
    "cleaned_transactions": PROCESSED_DATA_DIR / "cleaned_transactions.parquet",
    "invoice_items": PROCESSED_DATA_DIR / "invoice_items.parquet",
}
for path in input_paths.values():
    assert path.is_file(), f"Missing Phase 1 input: {path}"
for path in [TABLES_DIR, NETWORKS_DIR]:
    assert path.is_dir(), f"Missing output directory: {path}"
input_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in input_paths.items()}
validation_results = {}
selected_path = TABLES_DIR / "selected_products.csv"
pairs_path = TABLES_DIR / "pair_counts.csv"
graph_path = NETWORKS_DIR / "product_network.graphml"
summary_path = NETWORKS_DIR / "network_summary.json"


def distribution_summary(values):
    stats = pd.Series(values, dtype="float64").describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
    ).rename({"50%": "median"})
    return {name: None if pd.isna(value) else float(value) for name, value in stats.items()}

## 2. Input Validation
Chỉ kiểm tra hợp đồng đầu vào cần cho việc đếm cặp: các cột sử dụng, mã không thiếu, cặp hóa đơn–sản phẩm duy nhất và sự nhất quán giữa hai file. Không lặp lại kiểm tra Quantity, UnitPrice, hóa đơn hủy hoặc chuẩn hóa của Phase 1.

In [14]:
cleaned_transactions = pd.read_parquet(input_paths["cleaned_transactions"])
invoice_items = pd.read_parquet(input_paths["invoice_items"])
keys = ["InvoiceNo", "StockCode"]
assert set(keys + ["Description"]).issubset(cleaned_transactions.columns), "Missing transaction columns."
assert set(keys).issubset(invoice_items.columns) and not invoice_items.empty, "Invalid invoice-item input."
assert invoice_items[keys].notna().all().all() and not invoice_items.duplicated(keys).any(), "Missing or duplicate invoice-product keys."
assert pd.MultiIndex.from_frame(cleaned_transactions[keys]).unique().sort_values().equals(
    pd.MultiIndex.from_frame(invoice_items[keys]).sort_values()
), "Pair sets differ between Phase 1 inputs."
validation_results["input_pair_contract"] = True

basket_sizes = invoice_items.groupby("InvoiceNo").size()
large_invoice_ids = set(basket_sizes.index[basket_sizes.ge(100)])
input_statistics = {
    "cleaned_transaction_rows": len(cleaned_transactions),
    "invoice_item_rows": len(invoice_items),
    "invoices": int(invoice_items["InvoiceNo"].nunique()),
    "products": int(invoice_items["StockCode"].nunique()),
    "missing_customer_id_rows": int(cleaned_transactions["CustomerID"].isna().sum()),
    "missing_description_rows": int(cleaned_transactions["Description"].isna().sum()),
    "large_invoices_ge_100_products": len(large_invoice_ids),
    "basket_size": distribution_summary(basket_sizes),
}
display(pd.Series({key: value for key, value in input_statistics.items() if key != "basket_size"}, name="Input statistics"))
display(pd.Series(input_statistics["basket_size"], name="Distinct products per invoice"))

cleaned_transaction_rows          527746
invoice_item_rows                 517275
invoices                           19773
products                            3797
missing_customer_id_rows          131420
missing_description_rows               0
large_invoices_ge_100_products       636
Name: Input statistics, dtype: int64

count     19773.000000
mean         26.160674
std          47.019193
min           1.000000
25%           6.000000
median       15.000000
75%          29.000000
90%          52.000000
95%          75.000000
99%         218.000000
max        1106.000000
Name: Distinct products per invoice, dtype: float64

Basket size tính trên toàn bộ sản phẩm hợp lệ, trước Top N. Số cặp tăng theo `k*(k-1)/2`, nên hóa đơn lớn có thể đóng góp mạnh vào cấu trúc mạng; đây là lý do kiểm tra sensitivity, không phải bằng chứng dữ liệu sai.

## 3. Product Selection
Xếp hạng theo số hóa đơn phân biệt giảm dần, đồng hạng theo StockCode tăng dần. Nhãn là mô tả không thiếu, không rỗng, có chữ cái và xuất hiện nhiều nhất trong các dòng Phase 1; đồng hạng chọn Description tăng dần. Nếu không có mô tả hợp lệ, dùng StockCode làm nhãn và ghi cảnh báo. Không dùng Quantity hoặc doanh thu để chọn sản phẩm.

In [15]:
product_frequency = (
    invoice_items.groupby("StockCode")["InvoiceNo"].nunique().rename("invoice_count")
    .reset_index().sort_values(["invoice_count", "StockCode"], ascending=[False, True])
    .reset_index(drop=True)
)
selected_products = product_frequency.head(TOP_N_PRODUCTS).copy()
selected_codes = set(selected_products["StockCode"])
valid_label = (
    cleaned_transactions["StockCode"].isin(selected_codes)
    & cleaned_transactions["Description"].notna()
    & cleaned_transactions["Description"].str.strip().ne("")
    & cleaned_transactions["Description"].str.contains(r"[A-Za-z]", na=False)
)
label_counts = (
    cleaned_transactions.loc[valid_label].groupby(["StockCode", "Description"]).size()
    .rename("description_count").reset_index()
    .sort_values(["StockCode", "description_count", "Description"], ascending=[True, False, True])
)
labels = label_counts.drop_duplicates("StockCode")[["StockCode", "Description"]]
selected_products = selected_products.merge(labels, on="StockCode", how="left", validate="one_to_one", sort=False)
missing_label_codes = selected_products.loc[selected_products["Description"].isna(), "StockCode"].tolist()
selected_products["Description"] = selected_products["Description"].fillna(selected_products["StockCode"])
selected_products = selected_products[["StockCode", "Description", "invoice_count"]]
assert len(selected_products) == min(TOP_N_PRODUCTS, input_statistics["products"]) and selected_products["StockCode"].is_unique, "Invalid selected products."
validation_results["selected_products"] = True
display(selected_products.head(10))
display(pd.Series({
    "selected_products": len(selected_products),
    "minimum_invoice_count": int(selected_products["invoice_count"].min()),
    "maximum_invoice_count": int(selected_products["invoice_count"].max()),
    "fallback_labels": len(missing_label_codes),
}, name="Selection statistics"))

,StockCode,Description,invoice_count
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2265
1,85099B,JUMBO BAG RED RETROSPOT,2089
2,22423,REGENCY CAKESTAND 3 TIER,1988
3,47566,PARTY BUNTING,1685
4,20725,LUNCH BAG RED RETROSPOT,1565
5,84879,ASSORTED COLOUR BIRD ORNAMENT,1455
6,22197,POPCORN HOLDER,1392
7,22720,SET OF 3 CAKE TINS PANTRY DESIGN,1385
8,21212,PACK OF 72 RETROSPOT CAKE CASES,1320
9,22383,LUNCH BAG SUKI DESIGN,1284


selected_products         100
minimum_invoice_count     677
maximum_invoice_count    2265
fallback_labels             0
Name: Selection statistics, dtype: int64

## 4. Co-purchase Pair Construction
Tái sử dụng pair_counts.csv khi hash invoice_items không đổi và phạm vi sản phẩm đã lưu bao phủ các sản phẩm cần xét. Checksum trong summary xác nhận cache đã được kiểm tra; cache cũ chưa có checksum được đối chiếu chính xác một lần trước khi dùng lại.

Chỉ đếm lại baseline nếu không có cache phù hợp. Khi cần đếm, giữ nguyên cách sinh tổ hợp hai mã đã sắp xếp trong từng hóa đơn bằng Counter. Đóng góp của hóa đơn lớn được đếm riêng cho sensitivity; không tạo ma trận dense và không loại chúng khỏi baseline.

In [16]:
selected_items = invoice_items.loc[invoice_items["StockCode"].isin(selected_codes)].sort_values(["InvoiceNo", "StockCode"])
selected_baskets = selected_items.groupby("InvoiceNo")["StockCode"].agg(list)
cached_summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.is_file() else {}
cached_codes = set(pd.read_csv(selected_path, dtype={"StockCode": "string"})["StockCode"]) if selected_path.is_file() else set()
cache_inputs_match = cached_summary.get("input_files", {}).get("invoice_items", {}).get("sha256") == input_hashes["invoice_items"]
pair_counts_reused = pairs_path.is_file() and cache_inputs_match and selected_codes.issubset(cached_codes)
cache_info = cached_summary.get("pair_count_cache", {})
verified_cache = (
    pair_counts_reused
    and cache_info.get("invoice_items_sha256") == input_hashes["invoice_items"]
    and set(cache_info.get("stock_codes", [])) == cached_codes
    and cache_info.get("pair_counts_sha256") == hashlib.sha256(pairs_path.read_bytes()).hexdigest()
)
if pair_counts_reused:
    pair_counts = pd.read_csv(pairs_path, dtype={"ProductA": "string", "ProductB": "string", "pair_count": "int64"})
    pair_counts = pair_counts.loc[pair_counts["ProductA"].isin(selected_codes) & pair_counts["ProductB"].isin(selected_codes)].reset_index(drop=True)

pair_counter = Counter()
large_pair_counter = Counter()
for invoice, products in selected_baskets.items():
    if not pair_counts_reused:
        pair_counter.update(combinations(products, 2))
    if invoice in large_invoice_ids:
        large_pair_counter.update(combinations(products, 2))
if not pair_counts_reused:
    pair_counts = pd.DataFrame(
        [(a, b, count) for (a, b), count in sorted(pair_counter.items())],
        columns=["ProductA", "ProductB", "pair_count"],
    ).astype({"ProductA": "string", "ProductB": "string", "pair_count": "int64"})
selected_basket_sizes = selected_baskets.map(len)
assert not pair_counts.duplicated(["ProductA", "ProductB"]).any() and pair_counts["ProductA"].lt(pair_counts["ProductB"]).all(), "Duplicate, reversed or self pairs."
assert pd.api.types.is_integer_dtype(pair_counts["pair_count"]) and pair_counts["pair_count"].gt(0).all(), "Pair counts must be positive integers."
assert int(pair_counts["pair_count"].sum()) == int((selected_basket_sizes * (selected_basket_sizes - 1) // 2).sum()), "Pair total mismatch."
if not verified_cache:
    invoice_sets = {code: set(invoices) for code, invoices in selected_items.groupby("StockCode")["InvoiceNo"]}
    assert all(row.pair_count == len(invoice_sets[row.ProductA] & invoice_sets[row.ProductB]) for row in pair_counts.itertuples(index=False)), "Incorrect pair count."
validation_results.update({"canonical_positive_pairs": True, "exact_pair_counts_and_total": True})
full_pair_statistics = {
    "observed_pairs": len(pair_counts),
    "possible_pairs": len(selected_products) * (len(selected_products) - 1) // 2,
    "cooccurrence_events": int(pair_counts["pair_count"].sum()),
    "pair_count_distribution": distribution_summary(pair_counts["pair_count"]),
}
display(pd.Series({key: value for key, value in full_pair_statistics.items() if key != "pair_count_distribution"}, name="Full pairs before filtering"))
display(pair_counts.head())

observed_pairs           4950
possible_pairs           4950
cooccurrence_events    650269
Name: Full pairs before filtering, dtype: int64

,ProductA,ProductB,pair_count
0,20685,20712,102
1,20685,20719,70
2,20685,20723,64
3,20685,20724,90
4,20685,20725,125


Mỗi count là số hóa đơn chứa cả hai mã, không phải số lượng mua. `pair_counts.csv` giữ mọi cặp có count > 0; chỉ graph áp dụng ngưỡng cạnh.

## 5. Weighted Graph Construction
Tạo node cho mọi sản phẩm được chọn, gồm cả node cô lập. Chỉ thêm cạnh có `pair_count >= MIN_EDGE_WEIGHT`, với weight là số hóa đơn đồng mua. Hàm tạo graph được dùng lại trong sensitivity và không thay đổi graph chính.

In [17]:
def build_graph(products, counts, threshold):
    graph = nx.Graph()
    for row in products.itertuples(index=False):
        graph.add_node(row.StockCode, Description=row.Description, invoice_count=int(row.invoice_count))
    graph.add_weighted_edges_from(
        (row.ProductA, row.ProductB, int(row.pair_count))
        for row in counts.loc[counts["pair_count"].ge(threshold)].itertuples(index=False)
    )
    return graph


G = build_graph(selected_products, pair_counts, MIN_EDGE_WEIGHT)
assert set(G) == selected_codes and not G.is_directed() and nx.number_of_selfloops(G) == 0, "Invalid graph nodes or type."
expected_weights = {(row.ProductA, row.ProductB): int(row.pair_count) for row in pair_counts.itertuples(index=False) if row.pair_count >= MIN_EDGE_WEIGHT}
actual_weights = {tuple(sorted((a, b))): data["weight"] for a, b, data in G.edges(data=True)}
assert actual_weights == expected_weights, "Graph edges or weights do not match the threshold."
validation_results.update({"graph_nodes_type_no_self_loops": True, "graph_threshold_and_weights": True})
display(pd.Series({"nodes": G.number_of_nodes(), "edges": G.number_of_edges(), "edge_threshold": MIN_EDGE_WEIGHT}))

nodes              100
edges             4950
edge_threshold       3
dtype: int64

## 6. Network Diagnostics
Density là tỷ lệ cạnh thực tế trên số cặp node có thể có; components mô tả độ liên thông; degree đếm số láng giềng; edge weight đo tần suất đồng mua. So sánh graph của mọi cặp dương với graph sau lọc. Đây là kiểm tra cấu trúc, chưa phải phân tích community hoặc vai trò trung tâm.

In [18]:
def graph_metrics(graph):
    components = list(nx.connected_components(graph))
    return {
        "nodes": graph.number_of_nodes(),
        "edges": graph.number_of_edges(),
        "density": float(nx.density(graph)),
        "connected_components": len(components),
        "largest_component_size": max((len(component) for component in components), default=0),
        "isolated_nodes": nx.number_of_isolates(graph),
    }


full_graph = build_graph(selected_products, pair_counts, 1)
final_metrics = graph_metrics(G)
degree_summary = distribution_summary([degree for _, degree in G.degree()])
edge_weight_summary = distribution_summary([data["weight"] for _, _, data in G.edges(data=True)])
display(pd.DataFrame({"full_positive_pairs": graph_metrics(full_graph), "final_graph": final_metrics}))
display(pd.DataFrame({"degree": degree_summary, "edge_weight": edge_weight_summary}))

,full_positive_pairs,final_graph
nodes,100.0,100.0
edges,4950.0,4950.0
density,1.0,1.0
connected_components,1.0,1.0
largest_component_size,100.0,100.0
isolated_nodes,0.0,0.0


,degree,edge_weight
count,100.0,4950.000000
mean,99.0,131.367475
std,0.0,79.846886
min,99.0,19.000000
25%,99.0,81.000000
median,99.0,113.000000
75%,99.0,156.000000
90%,99.0,217.000000
95%,99.0,274.550000
99%,99.0,469.020000


Đọc bảng diagnostics để xác định ngưỡng hiện tại đã làm graph thưa đi hay chưa. Nếu density bằng 1, graph là complete graph và degree không có biến thiên giữa các node. Kết quả này cần được xem cùng sensitivity theo scope, threshold và hóa đơn lớn; không tự xác nhận tham số tối ưu.

## 7. Sensitivity Analysis
### A. Large invoices
Hóa đơn lớn có ≥100 sản phẩm trên **toàn bộ** invoice_items trước Top N. Scenario giữ nguyên danh sách sản phẩm baseline và chỉ trừ đóng góp của các hóa đơn này. Báo tỷ lệ đồng xuất hiện trên toàn bộ sản phẩm (chỉ tính số tổ hợp) và trong selected products. Graph chính không bị ghi đè.

Bổ sung baseline và scenario ở các ngưỡng 100, 140, 150 trên cùng phạm vi sản phẩm của graph chính. Tỷ lệ trọng số so với baseline dùng tổng weight còn lại của baseline tại chính ngưỡng đó. Các graph so sánh không thay đổi graph chính thức.

In [19]:
pair_changes = pair_counts.copy()
pair_changes["large_invoice_contribution"] = [large_pair_counter[(a, b)] for a, b in pair_counts[["ProductA", "ProductB"]].itertuples(index=False, name=None)]
pair_changes["scenario_pair_count"] = pair_changes["pair_count"] - pair_changes["large_invoice_contribution"]
scenario_pairs = pair_changes.loc[pair_changes["scenario_pair_count"].gt(0), ["ProductA", "ProductB", "scenario_pair_count"]].rename(columns={"scenario_pair_count": "pair_count"})
scenario_graph = build_graph(selected_products, scenario_pairs, MIN_EDGE_WEIGHT)
all_events = int((basket_sizes * (basket_sizes - 1) // 2).sum())
large_basket_sizes = basket_sizes.loc[basket_sizes.ge(100)]
large_all_events = int((large_basket_sizes * (large_basket_sizes - 1) // 2).sum())
selected_events = full_pair_statistics["cooccurrence_events"]
large_selected_events = sum(large_pair_counter.values())
large_invoice_summary = {
    "large_invoice_definition": "at least 100 distinct products before Top N",
    "excluded_invoices_in_scenario": len(large_invoice_ids),
    "all_product_cooccurrence_events": all_events,
    "large_invoice_all_product_events": large_all_events,
    "large_invoice_all_product_event_pct": 100 * large_all_events / all_events if all_events else 0.0,
    "selected_product_cooccurrence_events": selected_events,
    "large_invoice_selected_product_events": large_selected_events,
    "large_invoice_selected_product_event_pct": 100 * large_selected_events / selected_events if selected_events else 0.0,
    "pairs_changed": int(pair_changes["large_invoice_contribution"].gt(0).sum()),
    "pairs_lost": int(pair_changes["scenario_pair_count"].eq(0).sum()),
    "pair_count_decrease": distribution_summary(pair_changes["large_invoice_contribution"]),
    "baseline": {"invoices": input_statistics["invoices"], "pair_count_sum": selected_events, **final_metrics},
    "scenario": {"invoices": input_statistics["invoices"] - len(large_invoice_ids), "pair_count_sum": int(scenario_pairs["pair_count"].sum()), **graph_metrics(scenario_graph)},
}
display(pd.Series({key: value for key, value in large_invoice_summary.items() if not isinstance(value, dict)}, name="Large invoice sensitivity"))
display(pd.DataFrame({"baseline": large_invoice_summary["baseline"], "without_large_invoices": large_invoice_summary["scenario"]}))
display(pair_changes.sort_values(["large_invoice_contribution", "ProductA", "ProductB"], ascending=[False, True, True]).head(10))

large_threshold_rows = []
for threshold in [100, 140, 150]:
    baseline_weight = int(pair_counts.loc[pair_counts["pair_count"].ge(threshold), "pair_count"].sum())
    for case, counts in [("baseline", pair_counts), ("without_large_invoices", scenario_pairs)]:
        comparison_graph = build_graph(selected_products, counts, threshold)
        weights = [data["weight"] for _, _, data in comparison_graph.edges(data=True)]
        weight_stats = distribution_summary(weights)
        large_threshold_rows.append({
            "threshold": threshold, "case": case, **graph_metrics(comparison_graph),
            "edge_weight_sum": sum(weights),
            "edge_weight_mean": weight_stats["mean"],
            "edge_weight_median": weight_stats["median"],
            "edge_weight_vs_baseline_pct": 100 * sum(weights) / baseline_weight if baseline_weight else 0.0,
        })
large_threshold_sensitivity = pd.DataFrame(large_threshold_rows)
large_invoice_summary["threshold_comparisons"] = large_threshold_rows
display(large_threshold_sensitivity[[
    "threshold", "case", "edges", "density", "connected_components", "isolated_nodes",
    "edge_weight_sum", "edge_weight_median", "edge_weight_vs_baseline_pct",
]].round(3))

large_invoice_definition                    at least 100 distinct products before Top N
excluded_invoices_in_scenario                                                       636
all_product_cooccurrence_events                                                28363507
large_invoice_all_product_events                                               21650173
large_invoice_all_product_event_pct                                           76.331086
selected_product_cooccurrence_events                                             650269
large_invoice_selected_product_events                                            321499
large_invoice_selected_product_event_pct                                      49.440924
pairs_changed                                                                      4947
pairs_lost                                                                            0
Name: Large invoice sensitivity, dtype: object

,baseline,without_large_invoices
invoices,19773.0,19137.0
pair_count_sum,650269.0,328770.0
nodes,100.0,100.0
edges,4950.0,4950.0
density,1.0,1.0
connected_components,1.0,1.0
largest_component_size,100.0,100.0
isolated_nodes,0.0,0.0


,ProductA,ProductB,pair_count,large_invoice_contribution,scenario_pair_count
4090,22697,22699,767,217,550
4942,85099B,85123A,448,195,253
4480,22961,85099B,309,191,118
4098,22697,22961,255,186,69
4084,22666,85099B,285,184,101
3567,22457,85099B,319,183,136
4125,22697,85099B,270,183,87
3115,22379,85099B,415,181,234
4422,22960,22961,472,178,294
2865,22197,85099B,400,177,223


,threshold,case,edges,density,connected_components,isolated_nodes,edge_weight_sum,edge_weight_median,edge_weight_vs_baseline_pct
0,100,baseline,2976,0.601,1,0,506431,145.0,100.000
1,100,without_large_invoices,728,0.147,4,2,131969,147.5,26.059
2,140,baseline,1606,0.324,1,0,343840,182.0,100.000
3,140,without_large_invoices,402,0.081,18,13,93797,194.0,27.279
4,150,baseline,1375,0.278,1,0,310402,192.0,100.000
5,150,without_large_invoices,354,0.072,20,15,86846,212.5,27.979


### B. Edge threshold
Dùng lại pair_counts ở các ngưỡng 50, 100, 150, 200 và cấu hình hiện tại. Tỷ lệ trọng số giữ lại có mẫu số là tổng count của **mọi cặp dương trước lọc**. Không tự chọn lại threshold.

In [20]:
threshold_rows = []
for threshold in sorted({50, 100, 150, 200, MIN_EDGE_WEIGHT}):
    threshold_graph = build_graph(selected_products, pair_counts, threshold)
    retained_weight = sum(data["weight"] for _, _, data in threshold_graph.edges(data=True))
    threshold_rows.append({
        "threshold": threshold, **graph_metrics(threshold_graph),
        "retained_edge_weight": retained_weight,
        "total_edge_weight_retained_pct": 100 * retained_weight / selected_events if selected_events else 0.0,
    })
threshold_sensitivity = pd.DataFrame(threshold_rows)
display(threshold_sensitivity)

,threshold,nodes,edges,density,connected_components,largest_component_size,isolated_nodes,retained_edge_weight,total_edge_weight_retained_pct
0,3,100,4950,1.000000,1,100,0,650269,100.000000
1,50,100,4730,0.955556,1,100,0,641300,98.620725
2,100,100,2976,0.601212,1,100,0,506431,77.880231
3,150,100,1375,0.277778,1,100,0,310402,47.734399
4,200,100,619,0.125051,7,92,5,181156,27.858625


### C. Product scope × edge threshold
So sánh Top 50, 75, 100 tại các ngưỡng 100, 125, 140, 150, 175, 200 và MIN_EDGE_WEIGHT, chỉ dùng pair counts trong phạm vi đã tính. Bỏ qua scope chưa đủ dữ liệu; không mở rộng vượt Top 100. Giữ mọi selected node, kể cả isolates.

Invoice coverage là tỷ lệ hóa đơn có ít nhất một sản phẩm thuộc phạm vi trên toàn bộ hóa đơn hợp lệ. Product occurrence coverage là tỷ lệ cặp InvoiceNo–StockCode thuộc phạm vi trên toàn bộ invoice_items, không phải Quantity hoặc số dòng giao dịch. Coverage tính trước lọc cạnh, vẫn tính sản phẩm cô lập.

Retained weight có mẫu số là tổng pair counts dương của từng phạm vi. Tỷ lệ này không phải trọng số tuyệt đối để so trực tiếp giữa các scope khác nhau. Ba ứng viên minh họa các đánh đổi khác nhau; không xếp hạng, không tự chọn cấu hình thắng và không áp dụng vào config.

In [21]:
scope_rows = []
for scope in [50, 75, 100]:
    if len(selected_products) < scope:
        continue
    scope_products = selected_products.head(scope)
    scope_codes = set(scope_products["StockCode"])
    scope_items = invoice_items.loc[invoice_items["StockCode"].isin(scope_codes)]
    scope_pairs = pair_counts.loc[pair_counts["ProductA"].isin(scope_codes) & pair_counts["ProductB"].isin(scope_codes)]
    scope_total_weight = int(scope_pairs["pair_count"].sum())
    invoice_coverage = 100 * scope_items["InvoiceNo"].nunique() / input_statistics["invoices"]
    occurrence_coverage = 100 * len(scope_items) / len(invoice_items)
    for threshold in sorted({100, 125, 140, 150, 175, 200, MIN_EDGE_WEIGHT}):
        scope_graph = build_graph(scope_products, scope_pairs, threshold)
        retained_weight = sum(data["weight"] for _, _, data in scope_graph.edges(data=True))
        scope_rows.append({
            "top_n": scope, "threshold": threshold,
            "invoice_coverage_pct": invoice_coverage,
            "product_occurrence_coverage_pct": occurrence_coverage,
            **graph_metrics(scope_graph),
            "total_edge_weight_retained_pct": 100 * retained_weight / scope_total_weight if scope_total_weight else 0.0,
        })
scope_sensitivity = pd.DataFrame(scope_rows)
scope_status = "completed" if {row["top_n"] for row in scope_rows} == {50, 75, 100} else "partial" if scope_rows else "unavailable"
candidate_rows = []
if scope_rows:
    display(scope_sensitivity[[
        "top_n", "threshold", "invoice_coverage_pct", "product_occurrence_coverage_pct",
        "nodes", "edges", "density", "connected_components", "isolated_nodes",
        "total_edge_weight_retained_pct",
    ]].round(3))
    for scope, threshold, purpose, trade_off in [
        (50, 175, "compact", "Smaller graph and stronger threshold, but lower invoice and occurrence coverage."),
        (75, 140, "intermediate", "Intermediate scope and link retention, but still excludes less frequent products."),
        (100, 125, "broad", "Broader coverage and more retained links, but a larger graph with weaker pairs included."),
    ]:
        match = scope_sensitivity.loc[scope_sensitivity["top_n"].eq(scope) & scope_sensitivity["threshold"].eq(threshold)]
        if not match.empty:
            candidate_rows.append({"purpose": purpose, **match.to_dict("records")[0], "trade_off": trade_off, "applied": False})
candidate_configurations = pd.DataFrame(candidate_rows)
if candidate_rows:
    display(candidate_configurations[[
        "purpose", "top_n", "threshold", "invoice_coverage_pct", "product_occurrence_coverage_pct",
        "edges", "density", "connected_components", "isolated_nodes", "total_edge_weight_retained_pct",
    ]].round(3))

,top_n,threshold,invoice_coverage_pct,product_occurrence_coverage_pct,nodes,edges,density,connected_components,isolated_nodes,total_edge_weight_retained_pct
0,50,3,71.340,11.359,50,1225,1.000,1,0,100.000
1,50,100,71.340,11.359,50,1095,0.894,1,0,94.998
2,50,125,71.340,11.359,50,896,0.731,1,0,84.924
3,50,140,71.340,11.359,50,739,0.603,1,0,75.667
4,50,150,71.340,11.359,50,663,0.541,1,0,70.767
5,50,175,71.340,11.359,50,474,0.387,1,0,57.141
6,50,200,71.340,11.359,50,344,0.281,2,1,46.336
7,75,3,77.378,15.395,75,2775,1.000,1,0,100.000
8,75,100,77.378,15.395,75,2088,0.752,1,0,87.509
9,75,125,77.378,15.395,75,1570,0.566,1,0,73.834


,purpose,top_n,threshold,invoice_coverage_pct,product_occurrence_coverage_pct,edges,density,connected_components,isolated_nodes,total_edge_weight_retained_pct
0,compact,50,175,71.340,11.359,474,0.387,1,0,57.141
1,intermediate,75,140,77.378,15.395,1242,0.448,1,0,63.644
2,broad,100,125,80.625,18.860,2074,0.419,1,0,62.366


Các ứng viên đại diện cho ba hướng khảo sát, không phải kết quả tự động tối ưu. Phạm vi gọn giúp dễ trình bày nhưng giảm coverage; phạm vi trung gian giữ mức đại diện và số liên kết ở giữa; phạm vi rộng bao phủ nhiều lượt xuất hiện hơn nhưng có graph lớn và giữ thêm các cặp yếu hơn.

Cần xem đồng thời coverage, density, components, isolates và retained weight, thay vì chọn chỉ theo một chỉ số. Bảng large-invoice sensitivity áp dụng cho phạm vi đang cấu hình, không xác nhận độ vững riêng của từng ứng viên. Nếu scenario mất nhiều trọng số hoặc bị phân mảnh, cần nêu giới hạn này trong nghiên cứu tiếp theo. Baseline vẫn giữ toàn bộ hóa đơn hợp lệ.

## 8. Validation Summary
Các assert cốt lõi đặt ngay sau input, product selection, pair counting và graph construction. Pair counts được đối chiếu bằng giao của các tập hóa đơn và kiểm tra tổng số tổ hợp. Phần Outputs đọc lại CSV và GraphML để đối chiếu; không lặp lại validation preprocessing của Phase 1.

In [22]:
display(pd.Series(validation_results, name="Passed before saving"))

input_pair_contract               True
selected_products                 True
canonical_positive_pairs          True
exact_pair_counts_and_total       True
graph_nodes_type_no_self_loops    True
graph_threshold_and_weights       True
Name: Passed before saving, dtype: bool

## 9. Outputs & Phase 2 Summary
Export CSV và GraphML tương ứng với config hiện tại; chỉ ghi lại khi nội dung hoặc cấu trúc khác output cũ. Vì vậy thay đổi config ở một lần chạy sau không gây xung đột với file đã lưu. Các graph sensitivity và ứng viên không được ghi thành graph chính.

JSON summary lưu các bảng so sánh, danh sách ứng viên chưa áp dụng, chữ ký cache pair counts và validation. Input Phase 1 chỉ được đọc; không có bước sửa config hoặc tự loại hóa đơn lớn.

In [23]:
for frame, path in [(selected_products, selected_path), (pair_counts, pairs_path)]:
    csv_text = frame.to_csv(index=False)
    if not path.exists() or path.read_text(encoding="utf-8") != csv_text:
        path.write_text(csv_text, encoding="utf-8")
loaded_graph = nx.read_graphml(graph_path) if graph_path.is_file() else None
loaded_weights = {tuple(sorted((a, b))): data["weight"] for a, b, data in loaded_graph.edges(data=True)} if loaded_graph is not None else {}
if (loaded_graph is None or loaded_graph.is_directed()
        or dict(loaded_graph.nodes(data=True)) != dict(G.nodes(data=True)) or loaded_weights != actual_weights):
    nx.write_graphml(G, graph_path)
    loaded_graph = nx.read_graphml(graph_path)
    loaded_weights = {tuple(sorted((a, b))): data["weight"] for a, b, data in loaded_graph.edges(data=True)}

pd.testing.assert_frame_equal(pd.read_csv(selected_path, dtype={"StockCode": "string", "Description": "string"}), selected_products, check_dtype=False)
pd.testing.assert_frame_equal(pd.read_csv(pairs_path, dtype={"ProductA": "string", "ProductB": "string"}), pair_counts, check_dtype=False)
assert not loaded_graph.is_directed() and dict(loaded_graph.nodes(data=True)) == dict(G.nodes(data=True)), "GraphML nodes or type differ."
assert loaded_weights == actual_weights, "GraphML edges or weights differ."
assert all(hashlib.sha256(path.read_bytes()).hexdigest() == input_hashes[name] for name, path in input_paths.items()), "Phase 1 inputs changed."
validation_results.update({"csv_roundtrip": True, "graphml_roundtrip": True, "phase1_inputs_unchanged": True})

In [24]:
warnings = [
    "TOP_N_PRODUCTS and MIN_EDGE_WEIGHT are initial research settings, not validated optima.",
    f"Large invoices are retained in baseline; they contribute {large_invoice_summary['large_invoice_selected_product_event_pct']:.2f}% of selected-product cooccurrence events.",
]
if final_metrics["nodes"] > 1 and final_metrics["density"] == 1.0:
    warnings.append("The final graph is complete (density = 1): no positive pair is removed at the current threshold, and all nodes have identical degree. See threshold sensitivity before structural interpretation.")
if missing_label_codes:
    warnings.append(f"StockCode used as label for products without valid Description: {missing_label_codes}")
if scope_status != "completed":
    warnings.append("Top 50 / Top 75 / Top 100 comparison is incomplete because the calculated product scope is insufficient.")
if G.number_of_edges() == 0:
    warnings.append("No edge passes MIN_EDGE_WEIGHT; selected products remain as isolated nodes.")
summary = {
    "input_statistics": input_statistics,
    "input_files": {name: {"path": str(path.relative_to(PROJECT_ROOT)), "sha256": input_hashes[name]} for name, path in input_paths.items()},
    "configuration": {"TOP_N_PRODUCTS": TOP_N_PRODUCTS, "MIN_EDGE_WEIGHT": MIN_EDGE_WEIGHT, "RANDOM_STATE": RANDOM_STATE},
    "versions": {"python": sys.version.split()[0], "pandas": pd.__version__, "networkx": nx.__version__},
    "selected_product_statistics": {
        "products": len(selected_products), "invoices_with_selected_products": len(selected_baskets),
        "invoice_item_rows": len(selected_items),
        "invoice_count_distribution": distribution_summary(selected_products["invoice_count"]),
        "products_without_valid_description": missing_label_codes,
    },
    "full_pair_statistics": full_pair_statistics,
    "pair_count_cache": {
        "invoice_items_sha256": input_hashes["invoice_items"],
        "stock_codes": sorted(selected_codes),
        "pair_counts_sha256": hashlib.sha256(pairs_path.read_bytes()).hexdigest(),
    },
    "final_graph_metrics": {**final_metrics, "total_edge_weight": sum(actual_weights.values()), "degree_distribution": degree_summary, "edge_weight_distribution": edge_weight_summary},
    "validation_results": validation_results,
    "sensitivity_analysis": {
        "large_invoices": large_invoice_summary,
        "edge_thresholds": threshold_sensitivity.to_dict("records"),
        "product_scope": {
            "status": scope_status, "comparisons": scope_sensitivity.to_dict("records"),
            "invoice_coverage_denominator": input_statistics["invoices"],
            "product_occurrence_coverage_denominator": len(invoice_items),
            "coverage_basis": "Before edge filtering; invoices containing at least one scoped product and unique invoice-product occurrences, including isolated nodes.",
            "retained_weight_basis": "All positive pair counts within each product scope.",
            "candidates": candidate_rows,
            "candidate_basis": "Unranked illustrative configurations representing compact, intermediate and broad scopes; none is applied automatically.",
        },
    },
    "assumptions": [
        "InvoiceNo identifies an invoice; StockCode identifies a product.",
        "All valid Phase 1 invoices remain in baseline; customer IDs are not required.",
        "Product ranking uses distinct invoice counts with ascending StockCode tie-breaks.",
        "Description mode uses retained Phase 1 transaction rows, with ascending text tie-breaks; labels do not define identity.",
        "Each invoice contributes once per canonical product pair; counts precede edge filtering.",
        "Sensitivity scenarios keep the baseline product ranking fixed and do not modify the main graph.",
        "Threshold retained-weight percentages use all positive selected-product pair counts as denominator.",
        "No random sampling is used; sorted identifiers and tie-breaks ensure deterministic outputs.",
    ],
    "warnings": warnings,
    "output_files": [str(path.relative_to(PROJECT_ROOT)) for path in [selected_path, pairs_path, graph_path, summary_path]],
}
summary_text = json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False) + "\n"
if not summary_path.exists() or summary_path.read_text(encoding="utf-8") != summary_text:
    summary_path.write_text(summary_text, encoding="utf-8")
assert json.loads(summary_path.read_text(encoding="utf-8")) == summary, "JSON round-trip failed."
assert all(path.is_file() and path.stat().st_size > 0 for path in [selected_path, pairs_path, graph_path, summary_path])
display(pd.DataFrame({"output": summary["output_files"], "bytes": [path.stat().st_size for path in [selected_path, pairs_path, graph_path, summary_path]]}))

,output,bytes
0,outputs/tables/selected_products.csv,3781
1,outputs/tables/pair_counts.csv,77750
2,outputs/networks/product_network.graphml,431944
3,outputs/networks/network_summary.json,24068


Các output mô tả graph theo config hiện tại cùng ảnh hưởng của scope, threshold và hóa đơn lớn. Kết quả validation và các bảng trong notebook, JSON summary là bằng chứng cho lần chạy tương ứng.

Phase 2 nhằm xác định cấu hình đáng nghiên cứu tiếp, không khẳng định network tối ưu. Các ứng viên không làm thay đổi config hoặc baseline. Notebook chưa thực hiện community detection, centrality, modularity hoặc các chỉ số bán chéo.